#Arkavidia

In [55]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [56]:
import pandas as pd
import glob

path = '/content/drive/MyDrive/Dataset Arkavidia/ISPU/'
files = glob.glob(path + '*ispu*.csv')
simpan = []


for x in files:
  data = pd.read_csv(x, sep=None, engine='python')
  data.columns = data.columns.str.lower().str.strip()

  if 'tanggal' in data.columns and data['tanggal'].astype(str).str.len().max() <= 2:
        tahun_str = data['periode_data'].astype(str).str[:4]
        bulan_str = data['periode_data'].astype(str).str[4:6]
        hari_str = data['tanggal'].astype(str).str.zfill(2)

        data['tanggal'] = tahun_str + "-" + bulan_str + "-" + hari_str

  mapping = {
        'parameter_pencemar_kritis': 'critical',
        'parameter_kritis': 'critical',
        'pencemar_kritis': 'critical',
        'kategori': 'category',
        'categori': 'category',

        'pm_sepuluh': 'pm10', 'pm_10': 'pm10',
        'pm_duakomalima': 'pm25', 'pm_25': 'pm25',
        'lokasi_spku': 'stasiun',
        'sulfur_dioksida': 'so2', 'karbon_monoksida': 'co',
        'ozon': 'o3', 'nitrogen_dioksida': 'no2'
    }

  data.rename(columns=mapping, inplace=True)
  kolom_target = ['periode_data', 'tanggal', 'stasiun', 'pm10', 'pm25',
                    'so2', 'co', 'o3', 'no2', 'max', 'critical', 'category']

  ada_kolom = [y for y in kolom_target if y in data.columns]
  data = data[ada_kolom]
  simpan.append(data)



DataISPU = pd.concat(simpan,axis=0,ignore_index=True)
DataISPU['tanggal'] = pd.to_datetime(DataISPU['tanggal'],errors='coerce')
DataISPU = DataISPU[DataISPU['tanggal'].dt.year >= 2010]
DataISPU = DataISPU.dropna(subset=['tanggal'])
DataISPU = DataISPU.sort_values(by='tanggal').reset_index(drop=True)
print(f'Selesai GnG',{len(files)})
print(f"Rentang Data Akhir: {DataISPU['tanggal'].min()} sampai {DataISPU['tanggal'].max()}")


Selesai GnG {16}
Rentang Data Akhir: 2010-01-01 00:00:00 sampai 2025-08-31 00:00:00


/tmp/ipykernel_225/868663071.py:45: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  DataISPU['tanggal'] = pd.to_datetime(DataISPU['tanggal'],errors='coerce')


**EDA**

In [57]:
DataISPU.head()

,periode_data,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,category
0,201001,2010-01-01,DKI2 (Kelapa Gading),---,NaN,---,---,---,---,0,NaN,TIDAK ADA DATA
1,201001,2010-01-01,DKI3 (Jagakarsa),---,NaN,---,---,---,---,0,NaN,TIDAK ADA DATA
2,201001,2010-01-01,DKI5 (Kebon Jeruk),---,NaN,---,---,---,---,0,NaN,TIDAK ADA DATA
3,201001,2010-01-01,DKI1 (Bunderan HI),60,NaN,4,73,27,14,73,CO,SEDANG
4,201001,2010-01-01,DKI4 (Lubang Buaya),---,NaN,---,---,---,---,0,NaN,TIDAK ADA DATA


In [58]:
DataISPU.tail()

,periode_data,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,category
16896,202508,2025-08-31,DKI4 Lubang Buaya,47.0,59.0,27.0,10.0,18.0,17.0,59.0,PM25,SEDANG
16897,202508,2025-08-31,DKI1 Bundaran Hotel Indonesia HI,42.0,70.0,29.0,12.0,15.0,24.0,70.0,PM25,SEDANG
16898,202508,2025-08-31,DKI2 Kelapa Gading,NaN,72.0,45.0,16.0,21.0,16.0,72.0,PM25,SEDANG
16899,202508,2025-08-31,DKI5 Kebon Jeruk,37.0,65.0,25.0,9.0,21.0,34.0,65.0,PM25,SEDANG
16900,202508,2025-08-31,DKI3 Jagakarsa,28.0,60.0,53.0,8.0,19.0,39.0,60.0,PM25,SEDANG


In [59]:
DataISPU.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16901 entries, 0 to 16900
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   periode_data  16901 non-null  int64         
 1   tanggal       16901 non-null  datetime64[ns]
 2   stasiun       16901 non-null  object        
 3   pm10          16715 non-null  object        
 4   pm25          6944 non-null   object        
 5   so2           16848 non-null  object        
 6   co            16857 non-null  object        
 7   o3            16858 non-null  object        
 8   no2           16835 non-null  object        
 9   max           16894 non-null  object        
 10  critical      15410 non-null  object        
 11  category      16900 non-null  object        
dtypes: datetime64[ns](1), int64(1), object(10)
memory usage: 1.5+ MB


In [60]:
len(DataISPU)

16901

**DataCleaning**

In [61]:
polutan = ['pm10', 'so2', 'co', 'o3', 'no2']

for i in polutan:
  if i in DataISPU.columns:
    DataISPU[i] = pd.to_numeric(DataISPU[i], errors='coerce')

1. category harus di handle

In [62]:
DataISPU.isnull().sum()

,0
periode_data,0
tanggal,0
stasiun,0
pm10,2167
pm25,9957
so2,1772
co,1667
o3,1803
no2,1770
max,7


In [63]:
DataISPU['max'].value_counts()

,count
max,
0,1384
54,189
70,179
55,174
53,167
...,...
164,1
19,1
160,1


In [64]:
DataISPU['so2'].value_counts()

,count
so2,
27.0,426
14.0,424
28.0,411
15.0,410
26.0,408
...,...
80.0,1
89.0,1
77.0,1
